In [ ]:
# === auto-inserted: bev-solving src on path ===
import sys, pathlib
_root = pathlib.Path.cwd()
while _root != _root.parent and not (_root / 'src' / 'geometry.py').exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))


In [ ]:
from src.models.v5 import BEVDatasetV5, MultiCamBEVv5, build_rig_features, build_specialist_vocab, dump_v5_metadata
from src.splits import make_test_matched_split

"""
BEV v5 — calibration-aware + rover-specialist model.

What is new vs v4:
1. Explicit rig features from intrinsics/extrinsics, not only rover_id
2. Camera-wise FiLM on image features from per-camera calibration
3. BEV-wise FiLM from global rig summary + rover embedding
4. Specialist conditioning for top test rovers (shared trunk, specialised residual prior)
5. Test-matched group split helper
"""
from __future__ import annotations

import json
import math
import random
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

from bev_v1 import (
    BEV_H,
    BEV_RES,
    BEV_W,
    CAR2CAM_NAMES,
    CAMERA_NAMES,
    GT_NAME,
    INTRINSICS_NAMES,
    SmallUNet,
    X_RANGE,
    Y_RANGE,
    Z_LEVELS,
)
from bev_v4 import BEVDatasetV4, build_rover_vocab

def _camera_pose_features(car2cam: np.ndarray) -> np.ndarray:
    """
    We mostly need stable rig geometry. Translation is the strongest signal,
    and camera forward axis gives a compact orientation hint.
    """
    cam2car = np.linalg.inv(car2cam)
    trans = cam2car[:3, 3].astype(np.float32)
    forward = cam2car[:3, 2].astype(np.float32)
    return np.concatenate([trans, forward], axis=0)

class _ResNet34Backbone(nn.Module):
    def __init__(self, pretrained: bool = True):
        super().__init__()
        weights = torchvision.models.ResNet34_Weights.IMAGENET1K_V1 if pretrained else None
        rn = torchvision.models.resnet34(weights=weights)
        self.stem = nn.Sequential(rn.conv1, rn.bn1, rn.relu, rn.maxpool)
        self.layer1 = rn.layer1
        self.layer2 = rn.layer2
        self.proj = nn.Conv2d(128, 128, 1)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        return self.proj(x)

class _FiLM(nn.Module):
    def __init__(self, cond_dim: int, feat_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cond_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, feat_dim * 2),
        )

    def forward(self, cond: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        gamma_beta = self.net(cond)
        gamma, beta = torch.chunk(gamma_beta, 2, dim=-1)
        return gamma, beta


# Eval / threshold sweep / submission

Универсальный eval-ноутбук для `v1`, `v2`, `v4`, `v5`.

Что умеет:
- честно грузить `model` или `ema` state dict;
- работать с `official`, `group`, `test_matched` split;
- делать streaming threshold sweep;
- генерировать submission zip.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, json, zipfile, hashlib
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
import matplotlib.pyplot as plt

from bev_v1 import BEV_H, BEV_W
from bev_v3 import BEVDatasetAug, make_group_aware_split
from bev_v4 import BEVDatasetV4, build_rover_vocab
from bev_v5 import BEVDatasetV5, build_specialist_vocab, make_test_matched_split

DATA_TRAIN = Path('./autonomy_yandex_dataset_train/')
DATA_VAL   = Path('./autonomy_yandex_dataset_val/')
DATA_TEST  = Path('./autonomy_yandex_dataset_test/')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

Unknown instance spec: Please select VM configuration

## Config

In [ ]:
cfg = {
    'run_dir': './runs/v5',
    'which_ckpt': 'ema',        # 'best' | 'ema' | 'last'
    'model_type': 'v5',        # 'v1' | 'v2' | 'v4' | 'v5'
    'state_key': 'auto',       # 'auto' | 'model' | 'ema'
    'img_hw': (384, 704),
    'batch_size': 4,
    'num_workers': 4,
    'split_mode': 'test_matched',  # 'official' | 'group' | 'test_matched'
    'sweep_min': 0.20,
    'sweep_max': 0.70,
    'sweep_step': 0.02,
    'infer_threshold_override': None,
    'make_submission': False,
    'submission_name': None,
}

RUN_DIR = Path(cfg['run_dir'])
CKPT_PATH = RUN_DIR / f"{cfg['which_ckpt']}.pt"
assert CKPT_PATH.exists(), CKPT_PATH
print(json.dumps({k: str(v) for k, v in cfg.items()}, indent=2))

## Model / dataset factory

In [ ]:
def make_model_and_dataset_factory(cfg):
    img_hw = cfg['img_hw']

    if cfg['model_type'] == 'v1':
        from bev_v1 import BEVDataset, MultiCamBEV
        model = MultiCamBEV(freeze_backbone=False)
        def make_dataset(data_dir, mode):
            return BEVDataset(data_dir, mode=mode, img_hw=img_hw)
        return model, make_dataset

    if cfg['model_type'] == 'v2':
        from bev_v2 import MultiCamBEVPretrainedEncoder
        model = MultiCamBEVPretrainedEncoder(load_pretrained=False, freeze_encoder=False)
        def make_dataset(data_dir, mode):
            return BEVDatasetAug(data_dir, mode=mode, img_hw=img_hw, aug=False)
        return model, make_dataset

    if cfg['model_type'] == 'v4':
        from bev_v4 import MultiCamBEVv4
        rover_vocab = build_rover_vocab(DATA_TRAIN/'info.csv', DATA_VAL/'info.csv', DATA_TEST/'info.csv')
        model = MultiCamBEVv4(num_rovers=len(rover_vocab))
        def make_dataset(data_dir, mode):
            return BEVDatasetV4(data_dir, mode=mode, img_hw=img_hw, aug=False, rover_vocab=rover_vocab)
        return model, make_dataset

    if cfg['model_type'] == 'v5':
        from bev_v5 import MultiCamBEVv5
        rover_vocab = build_rover_vocab(DATA_TRAIN/'info.csv', DATA_VAL/'info.csv', DATA_TEST/'info.csv')
        specialist_vocab = build_specialist_vocab(DATA_TRAIN/'info.csv', DATA_TEST/'info.csv')
        model = MultiCamBEVv5(num_rovers=len(rover_vocab), num_specialists=len(specialist_vocab))
        def make_dataset(data_dir, mode):
            return BEVDatasetV5(
                data_dir, mode=mode, img_hw=img_hw, aug=False,
                rover_vocab=rover_vocab, specialist_vocab=specialist_vocab,
            )
        return model, make_dataset

    raise ValueError(cfg['model_type'])


def resolve_state_dict(ckpt, which_ckpt='best', state_key='auto'):
    if state_key == 'model':
        return ckpt['model'], 'model'
    if state_key == 'ema':
        return ckpt['ema'], 'ema'
    if which_ckpt == 'ema' and 'ema' in ckpt:
        return ckpt['ema'], 'ema'
    return ckpt['model'], 'model'


def model_forward(model, batch, model_type):
    imgs = batch['images'].to(device, non_blocking=True)
    intr = batch['intrinsics'].to(device, non_blocking=True)
    c2c = batch['car2cams'].to(device, non_blocking=True)

    if model_type in ['v1', 'v2']:
        return model(imgs, intr, c2c)
    if model_type == 'v4':
        rov = batch['rover_id'].to(device, non_blocking=True)
        return model(imgs, intr, c2c, rov)

    rov = batch['rover_id'].to(device, non_blocking=True)
    cam_rig = batch['cam_rig'].to(device, non_blocking=True)
    global_rig = batch['global_rig'].to(device, non_blocking=True)
    specialist = batch['specialist_id'].to(device, non_blocking=True)
    return model(imgs, intr, c2c, rov, cam_rig, global_rig, specialist)

## Load checkpoint

In [ ]:
model, make_dataset = make_model_and_dataset_factory(cfg)
ckpt = torch.load(CKPT_PATH, map_location=device)
state_dict, used_state_key = resolve_state_dict(
    ckpt,
    which_ckpt=cfg['which_ckpt'],
    state_key=cfg['state_key'],
)
missing, unexpected = model.load_state_dict(state_dict, strict=False)
model = model.to(device).eval()

print(json.dumps({
    'ckpt_path': str(CKPT_PATH),
    'used_state_key': used_state_key,
    'missing_keys': len(missing),
    'unexpected_keys': len(unexpected),
    'meta': {
        k: ckpt.get(k) for k in [
            'epoch', 'val_iou', 'val_iou_off',
            'ema_val_iou', 'ema_val_iou_off',
            'best_iou', 'best_ema_iou',
        ] if k in ckpt
    },
}, indent=2, default=str))

## Val loader

In [ ]:
if cfg['split_mode'] == 'official':
    val_ds = make_dataset(DATA_VAL, mode='val')
    split_name = 'official'
elif cfg['split_mode'] == 'group':
    _, val_idx = make_group_aware_split(
        DATA_TRAIN / 'info.csv',
        group_cols=('rover', 'ride_date'),
        holdout_frac=0.2,
        seed=42,
        cache_path=RUN_DIR / 'group_split.npz',
    )
    val_ds = Subset(make_dataset(DATA_TRAIN, mode='val'), val_idx)
    split_name = 'group'
else:
    _, val_idx = make_test_matched_split(
        DATA_TRAIN / 'info.csv',
        DATA_TEST / 'info.csv',
        holdout_frac=0.2,
        seed=42,
        cache_path=RUN_DIR / 'test_matched_split.npz',
    )
    val_ds = Subset(make_dataset(DATA_TRAIN, mode='val'), val_idx)
    split_name = 'test_matched'

val_loader = DataLoader(
    val_ds,
    batch_size=cfg['batch_size'],
    shuffle=False,
    num_workers=cfg['num_workers'],
    pin_memory=True,
)

print(split_name, len(val_ds), 'samples')

## Streaming threshold sweep

In [ ]:
from src.metrics import streaming_threshold_sweep

thresholds = np.arange(cfg['sweep_min'], cfg['sweep_max'] + 1e-9, cfg['sweep_step']).round(4).tolist()
iou_by_t = streaming_threshold_sweep(model, val_loader, thresholds, cfg['model_type'])
best_t, best_iou = max(iou_by_t.items(), key=lambda kv: kv[1])
print(f'best threshold: {best_t:.3f} | IoU={best_iou:.4f}')
for t, iou in iou_by_t.items():
    marker = ' <- best' if t == best_t else ''
    print(f't={t:.3f}  iou={iou:.4f}{marker}')


In [ ]:
ts = list(iou_by_t.keys())
ious = [iou_by_t[t] for t in ts]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ts, ious, marker='o')
ax.axvline(best_t, color='red', linestyle='--', alpha=0.5, label=f'best={best_t:.3f}')
ax.grid(); ax.legend(); ax.set_xlabel('threshold'); ax.set_ylabel('IoU')
plt.tight_layout(); plt.show()

## Test inference + submission

In [ ]:
infer_t = cfg['infer_threshold_override'] if cfg['infer_threshold_override'] is not None else best_t
print('using infer threshold:', infer_t)

if cfg['make_submission']:
    test_ds = make_dataset(DATA_TEST, mode='test')
    test_loader = DataLoader(test_ds, batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'])
    test_info = pd.read_csv(DATA_TEST / 'info.csv', index_col=0)
    out_dir = DATA_TEST / 'predicted_static_grids'
    out_dir.mkdir(exist_ok=True)

    model.eval()
    with torch.no_grad():
        for batch in tqdm(test_loader, desc='test inference'):
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model_forward(model, batch, cfg['model_type']).float()
            preds = (torch.sigmoid(logits) > infer_t).cpu().numpy().astype(np.int32)
            idxs = batch['info_idx']
            for k, idx in enumerate(idxs):
                out_path = test_info.iloc[idx.item()]['predicted_occupancy_grid']
                np.save(out_path, preds[k].reshape(1, BEV_H, BEV_W))

    sub_name = cfg['submission_name'] or f"submission_{RUN_DIR.name}_{cfg['model_type']}_{cfg['which_ckpt']}.zip"
    sub_path = Path(sub_name)
    if sub_path.exists():
        sub_path.unlink()

    with zipfile.ZipFile(sub_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        zf.write(DATA_TEST / 'info.csv', arcname='info.csv')
        for npy in sorted((DATA_TEST / 'predicted_static_grids').glob('*.npy')):
            zf.write(npy, arcname=f'predicted_static_grids/{npy.name}')

    with zipfile.ZipFile(sub_path, 'r') as zf:
        bad = zf.testzip()
        assert bad is None, bad

    h = hashlib.sha256()
    with open(sub_path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)

    print(json.dumps({
        'submission': str(sub_path.resolve()),
        'size_mb': round(sub_path.stat().st_size / 1e6, 3),
        'sha256': h.hexdigest(),
    }, indent=2))
else:
    print('cfg[\'make_submission\'] = False, inference skipped')